<a href="https://colab.research.google.com/github/j-see17/Digital-Analytics-Digital-Twin-/blob/main/GROUP_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_csv(list(uploaded.keys())[0])

df.head()

Saving real_data.csv to real_data (4).csv


,startDate,endDate,status,ipAddress,progress,duration,finished,recordedDate,_recordId,recipientLastName,...,QID19_5,QID20,QID20_6_TEXT,QID12,QID12_5_TEXT,QID21,QID21_5_TEXT,QID22,QID23,Q_DataPolicyViolations
0,Start Date,End Date,Response Type,IP Address,Progress,Duration (in seconds),Finished,Recorded Date,Response ID,Recipient Last Name,...,Q8_5 - How likely would you be to report your ...,Q9 - Which ONE incentive would most motivate y...,Q9_6_TEXT - Which ONE incentive would most mot...,Q10 - What would be your preferred way to repo...,Q10_5_TEXT - What would be your preferred way ...,Q11 - How do you prefer to receive reminders f...,Q11_5_TEXT - How do you prefer to receive remi...,Q12 - What is your current academic year?,Q13 - What is your major?,Q_DataPolicyViolations
1,3/11/2026 15:46,3/11/2026 15:46,Survey Preview,NaN,100,2,TRUE,3/11/2026 15:46,R_5ehsWLXi3JZObvB,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3/30/2026 14:03,3/30/2026 14:06,IP Address,129.2.89.203,100,149,TRUE,3/30/2026 14:06,R_7SqXAMDlm7EQFgd,NaN,...,Somewhat likely,Reminder emails,NaN,Online form,NaN,Text message,NaN,NaN,Business,NaN
3,3/30/2026 20:13,3/30/2026 20:16,IP Address,173.66.33.132,100,145,TRUE,3/30/2026 20:16,R_6HNdg8l6UY3zPms,NaN,...,Somewhat likely,Reminder emails,NaN,Online form,NaN,Email,NaN,Senior,Business,NaN
4,3/31/2026 10:07,3/31/2026 10:07,IP Address,129.2.89.7,100,4,TRUE,3/31/2026 10:07,R_5JxVZ5N2YsYZYsU,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
df.shape
df.columns

Index(['startDate', 'endDate', 'status', 'ipAddress', 'progress', 'duration',
       'finished', 'recordedDate', '_recordId', 'recipientLastName',
       'recipientFirstName', 'recipientEmail', 'externalDataReference',
       'locationLatitude', 'locationLongitude', 'distributionChannel',
       'userLanguage', 'QID13', 'QID24_TEXT', 'QID3', 'QID14', 'QID15_1',
       'QID15_2', 'QID15_3', 'QID15_4', 'QID15_5', 'QID15_6', 'QID15_7',
       'QID15_8', 'QID15_8_TEXT', 'QID16_1', 'QID16_2', 'QID16_3', 'QID16_4',
       'QID17_1', 'QID17_2', 'QID17_3', 'QID17_4', 'QID18_TEXT', 'QID19_1',
       'QID19_2', 'QID19_3', 'QID19_4', 'QID19_5', 'QID20', 'QID20_6_TEXT',
       'QID12', 'QID12_5_TEXT', 'QID21', 'QID21_5_TEXT', 'QID22', 'QID23',
       'Q_DataPolicyViolations'],
      dtype='object')

In [29]:
df.describe(include='all')

#explored the pilot dataset structure

,startDate,endDate,status,ipAddress,progress,duration,finished,recordedDate,_recordId,recipientLastName,...,QID19_5,QID20,QID20_6_TEXT,QID12,QID12_5_TEXT,QID21,QID21_5_TEXT,QID22,QID23,Q_DataPolicyViolations
count,15,15,15,14,15,15,15,15,15,1,...,8,8,1,8,1,8,1,7,8,1
unique,15,15,3,14,2,14,2,15,15,1,...,3,5,1,2,1,3,1,3,3,1
top,Start Date,End Date,IP Address,IP Address,100,3,TRUE,Recorded Date,Response ID,Recipient Last Name,...,Somewhat likely,Reminder emails,Q9_6_TEXT - Which ONE incentive would most mot...,Online form,Q10_5_TEXT - What would be your preferred way ...,Email,Q11_5_TEXT - How do you prefer to receive remi...,Senior,Business,Q_DataPolicyViolations
freq,1,1,13,1,14,2,14,1,1,1,...,6,2,1,7,1,6,1,5,6,1


In [39]:
# Drop the first row which contains descriptive headers that were read as data
df = df.iloc[1:].copy()

df['startDate'] = pd.to_datetime(df['startDate'], errors='coerce')
df['endDate'] = pd.to_datetime(df['endDate'], errors='coerce')

df['completion_minutes'] = (
    df['endDate'] - df['startDate']
).dt.total_seconds()/60

In [41]:
df['timing_group'] = pd.cut(
    df['completion_minutes'],
    bins=[0,5,10,20,100],
    labels=['fast','medium','slow','very_slow']
)

df['timing_group'].value_counts()


#timing strata identification

,count
timing_group,
fast,5
medium,1
very_slow,1
slow,0


In [43]:
#engagement : open ended response length
df['text_length'] = df.iloc[:,0].astype(str).str.len()


#shows engagement levels
df['engagement'] = pd.cut(
    df['text_length'],
    bins=[0,30,80,500],
    labels=['low','medium','high']
)

df['engagement'].value_counts()

,count
engagement,
low,12
medium,0
high,0


In [46]:
#sentiment strata
!pip install textblob

from textblob import TextBlob

df['sentiment'] = df.iloc[:,0].astype(str).apply(
    lambda x: TextBlob(x).sentiment.polarity
)

#sentiment analysis requirement:

df['sentiment_group'] = pd.cut(
    df['sentiment'],
    bins=[-1,-0.1,0.1,1],
    labels=['negative','neutral','positive']
)

df['sentiment_group'].value_counts()

,count
sentiment_group,
neutral,12
negative,0
positive,0


In [47]:
df['sentiment_group'] = pd.cut(
    df['sentiment'],
    bins=[-1,-0.1,0.1,1],
    labels=['negative','neutral','positive']
)

df['sentiment_group'].value_counts()

,count
sentiment_group,
neutral,12
negative,0
positive,0


In [48]:
#pilot proportions
pilot_strata = df.groupby(
    ['timing_group','engagement','sentiment_group']
).size()

pilot_strata = pilot_strata / pilot_strata.sum()

pilot_strata

#probabilities used to generate synthetic respondents

/tmp/ipykernel_21375/617370726.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pilot_strata = df.groupby(


timing_group  engagement  sentiment_group
fast          low         negative           0.000000
                          neutral            0.714286
                          positive           0.000000
              medium      negative           0.000000
                          neutral            0.000000
                          positive           0.000000
              high        negative           0.000000
                          neutral            0.000000
                          positive           0.000000
medium        low         negative           0.000000
                          neutral            0.142857
                          positive           0.000000
              medium      negative           0.000000
                          neutral            0.000000
                          positive           0.000000
              high        negative           0.000000
                          neutral            0.000000
                          positive           0.000000
slow          low         negative           0.000000
                          neutral            0.000000
                          positive           0.000000
              medium      negative           0.000000
                          neutral            0.000000
                          positive           0.000000
              high        negative           0.000000
                          neutral            0.000000
                          positive           0.000000
very_slow     low         negative           0.000000
                          neutral            0.142857
                          positive           0.000000
              medium      negative           0.000000
                          neutral            0.000000
                          positive           0.000000
              high        negative           0.000000
                          neutral            0.000000
                          positive           0.000000
dtype: float64

In [49]:
import numpy as np

N = 1000
synthetic = []

def generate_likert():
    r = np.random.rand()

    if r < 0.30:
        return np.random.choice([1,5])
    elif r < 0.50:
        return 3
    else:
        return np.random.choice([2,4])

for i in range(N):

    strata = pilot_strata.sample(weights=pilot_strata).index[0]

    timing, engagement, sentiment = strata

    row = {
        "timing_group": timing,
        "engagement": engagement,
        "sentiment_group": sentiment,
        "Q_value": generate_likert(),
        "completion_minutes": np.random.normal(
            df['completion_minutes'].mean(),
            df['completion_minutes'].std()
        )
    }

    synthetic.append(row)

synthetic_df = pd.DataFrame(synthetic)

synthetic_df.head()

,timing_group,engagement,sentiment_group,Q_value,completion_minutes
0,fast,low,neutral,2,-107.178372
1,fast,low,neutral,1,-447.594494
2,fast,low,neutral,3,-186.440211
3,fast,low,neutral,5,-211.651821
4,fast,low,neutral,1,-321.475954


In [51]:
synthetic_df.to_csv(
    "synthetic_digitaltwin_pooled.csv",
    index=False
)

files.download("synthetic_digitaltwin_pooled.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>